# Auto encoder

An Autoencoder for anomaly detection learns to reconstruct normal data.

`Previous chapter we learn auto_encoder in Dimensionality reduction . `

So the main question is that , auto_encoder in dimensionality and auto_encoder in anomaly both are same thing ? 

So the answer is :- 

| Autoencoder                              | Main purpose        |
| ---------------------------------------- | ------------------- |
| **Dimensionality Reduction Autoencoder** | Compress data       |
| **Anomaly Detection Autoencoder**        | Detect unusual data |


# For more info click the link below

# Notes and interview question and answer

# Coding

In [1]:
import numpy as np                                      # Numerical calculations

import pandas as pd                                     # Data manipulation

import matplotlib.pyplot as plt                         # Visualization

from sklearn.datasets import load_breast_cancer        # Breast Cancer dataset

from sklearn.model_selection import train_test_split    # Split data into training/testing sets

from sklearn.preprocessing import StandardScaler       # Standardize numerical features

from sklearn.metrics import classification_report       # Evaluate anomaly detection

import tensorflow as tf                                  # Deep learning framework

from tensorflow.keras import Model                       # Build the Autoencoder

from tensorflow.keras import Input                       # Define model input

from tensorflow.keras.layers import Dense                # Fully connected neural-network layer

# Dataset


In [2]:
data = load_breast_cancer()                              # Load Breast Cancer dataset

X = data.data                                             # Store the 30 numerical features

y = data.target                                           # Store the class labels

print("X shape:", X.shape)                                # Display number of samples and features

print("Classes:", data.target_names)                      # Display class names

X shape: (569, 30)
Classes: ['malignant' 'benign']


# Separate normal and anomaly samples 


In [3]:
X_normal = X[y == 1]                                     # Select benign samples as normal data

X_anomaly = X[y == 0]                                    # Select malignant samples as anomalies

print("Normal samples:", X_normal.shape[0])              # Number of normal samples

print("Anomaly samples:", X_anomaly.shape[0])            # Number of anomalous samples

Normal samples: 357
Anomaly samples: 212


# Split normal data 

`Important : We don't want the autoencoder to learn the anomalies . `

In [4]:
X_normal_train, X_normal_test = train_test_split(
    X_normal,                                             # Use only normal samples
    test_size=0.2,                                       # Keep 20% for testing
    random_state=42                                      # Make the split reproducible
)


#Normal data
#     ↓
#┌───────────────┐
#│ Normal Train  │ → Train Autoencoder
#└───────────────┘
#
#┌───────────────┐
#│ Normal Test   │ → Test reconstruction
#└───────────────┘

#Malignant
#     ↓
#Anomalies → Test reconstruction

# Scale the features

This is particularly important for an Autoencoder because the neural network should not be dominated by features with larger numerical ranges.

In [5]:
scaler = StandardScaler()                                 # Create standardization object

X_normal_train_scaled = scaler.fit_transform(
    X_normal_train                                        # Learn scaling parameters from normal training data
)

X_normal_test_scaled = scaler.transform(
    X_normal_test                                         # Apply the same scaling to normal test data
)

X_anomaly_scaled = scaler.transform(
    X_anomaly                                             # Apply the same scaling to anomalous data
)


#fit_transform()

#→ Training data only
#
#transform()
#→ Test/anomaly data

# Build the auto encoder 

we have :

In [6]:
# 30 input features
#       ↓
# 16 neurons
#       ↓
# 8 neurons
#        ↓
# 4 neurons
#        ↓
# 8 neurons
#       ↓
# 16 neurons
#       ↓
# 30 output features

# The middle 4 is the bottleneck


input_layer = Input(
    shape=(X_normal_train_scaled.shape[1],)              # Input has 30 features
)


encoder = Dense(
    16,
    activation="relu"                                    # Compress 30 features into 16
)(input_layer)


encoder = Dense(
    8,
    activation="relu"                                    # Compress 16 features into 8
)(encoder)


latent = Dense(
    4,
    activation="relu"                                    # Bottleneck representation with 4 features
)(encoder)


decoder = Dense(
    8,
    activation="relu"                                    # Expand 4 features back to 8
)(latent)


decoder = Dense(
    16,
    activation="relu"                                    # Expand 8 features back to 16
)(decoder)


output_layer = Dense(
    30,
    activation="linear"                                  # Reconstruct the original 30 features
)(decoder)


autoencoder = Model(
    input_layer,
    output_layer                                         # Complete encoder + decoder
)

# Compile the autoencoder

In [7]:
autoencoder.compile(
    optimizer="adam",                                    # Adam updates the neural-network weights
    loss="mse"                                           # MSE measures reconstruction error
)

# Why MSE?

# Because we're comparing:

# Original feature values
#        vs
# Reconstructed feature values
#
# Mathematically:
#
# MSE = average((original - reconstructed)²)

# Train it 

Here is the important part

We train only on normal data.


In [ ]:
history = autoencoder.fit(
    X_normal_train_scaled,                               # Train only on normal samples
    X_normal_train_scaled,                               # Target is the original normal data
    epochs=100,                                          # Train for 100 passes through the data
    batch_size=32,                                       # Process 32 samples at a time
    validation_split=0.2,                                # Use 20% of training data for validation
    shuffle=True,                                        # Shuffle training samples
    verbose=1                                            # Display training progress
)

Epoch 1/100


# plot training loss

In [ ]:
plt.figure(figsize=(8, 5))                              # Create the figure

plt.plot(
    history.history["loss"],                            # Training reconstruction loss
    label="Training Loss"
)

plt.plot(
    history.history["val_loss"],                        # Validation reconstruction loss
    label="Validation Loss"
)

plt.xlabel("Epoch")                                     # X-axis label

plt.ylabel("MSE Loss")                                  # Y-axis label

plt.title("Autoencoder Training Loss")                  # Chart title

plt.legend()                                            # Show legend

plt.show()                                              # Display the chart

# Calculate the reconstructions errors . 

`Now comes the core of anomaly detection `

`First  reconstruct normal samples.`

In [ ]:
normal_reconstructed = autoencoder.predict(
    X_normal_test_scaled,                                # Normal test data
    verbose=0                                            # Hide prediction progress
)

`Calculate the error :`

In [ ]:
normal_errors = np.mean(
    np.square(
        X_normal_test_scaled - normal_reconstructed       # Difference between original and reconstructed data
    ),
    axis=1                                               # Calculate one error value per sample
)

# Now do the same for anomalies : 


In [ ]:
anomaly_reconstructed = autoencoder.predict(
    X_anomaly_scaled,                                    # Anomalous samples
    verbose=0                                            # Hide prediction progress
)

In [ ]:
anomaly_errors = np.mean(
    np.square(
        X_anomaly_scaled - anomaly_reconstructed          # Reconstruction difference
    ),
    axis=1                                               # One error value per anomaly
)

# Chose an anomaly threshold 
`This is one of the most importatn parts .`

`We will use 95th percentile of normal reconstruction errors. `

In [ ]:
threshold = np.percentile(
    normal_errors,                                       # Reconstruction errors from normal samples
    95                                                    # 95th percentile becomes the threshold
)

print("Anomaly threshold:", threshold)

# project anomalies 

`Combine normal and anomalous samples: `

In [ ]:
all_errors = np.concatenate(
    [
        normal_errors,                                    # Errors from normal samples
        anomaly_errors                                    # Errors from anomalous samples
    ]
)

# Create the true labels : 

In [ ]:
y_true = np.concatenate(
    [
        np.zeros(len(normal_errors)),                      # 0 = normal
        np.ones(len(anomaly_errors))                       # 1 = anomaly
    ]
)

# Predict: 

In [ ]:
y_pred = (
    all_errors > threshold                                # Error above threshold = anomaly
).astype(int)

# Evaluate 

In [ ]:
print(
    classification_report(
        y_true,                                            # Actual normal/anomaly labels
        y_pred,                                            # Predicted normal/anomaly labels
        target_names=["Normal", "Anomaly"]                # Names for the two classes
    )
)

# Visualize reconstructions errors. 

In [ ]:
plt.figure(figsize=(10, 5))                              # Create the figure

plt.hist(
    normal_errors,
    bins=30,
    alpha=0.7,
    label="Normal"                                       # Distribution of normal errors
)

plt.hist(
    anomaly_errors,
    bins=30,
    alpha=0.7,
    label="Anomaly"                                      # Distribution of anomaly errors
)

plt.axvline(
    threshold,
    linestyle="--",
    label="Threshold"                                    # Display anomaly threshold
)

plt.xlabel("Reconstruction Error")                       # X-axis label

plt.ylabel("Number of Samples")                          # Y-axis label

plt.title("Normal vs Anomaly Reconstruction Error")     # Chart title

plt.legend()                                             # Show legend

plt.show()                                               # Display chart